In [1]:
!pip install -q transformers accelerate sentence-transformers chromadb pypdf

In [2]:
!pip install -U transformers

In [3]:
from google.colab import files
uploaded = files.upload()

Saving UNIT III BE.pdf to UNIT III BE (1).pdf


In [4]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

# Load PDF
loader = PyPDFLoader("UNIT III BE.pdf")
docs = loader.load()

# Chunking
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(docs)

# Embeddings (FREE)
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Store in DB
db = Chroma.from_documents(chunks, embeddings)

# Retriever (IMPORTANT — this is what I meant by “keep same”)
retriever = db.as_retriever(search_kwargs={"k": 3})

print("✅ PDF processed")

/tmp/ipykernel_46830/4127482199.py:15: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ PDF processed


In [6]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")

def generator(prompt, do_sample=False, max_new_tokens=80):
    inputs = tokenizer(prompt, return_tensors="pt")
    outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=do_sample)
    return [{"generated_text": tokenizer.decode(outputs[0], skip_special_tokens=True)}]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [10]:
import re

def clean_text(text):
    # remove weird symbols
    text = re.sub(r'[^a-zA-Z0-9\s.,]', ' ', text)

    # remove repeated characters
    text = re.sub(r'(.)\1{3,}', '', text)

    # remove extra spaces
    text = re.sub(r'\s+', ' ', text)

    return text.strip()


def is_valid(text):
    # keep only meaningful chunks
    words = text.split()
    return len(words) > 20   # filter small/garbage chunks


def ask(query):
    docs = retriever.invoke(query)

    if not docs:
        return None, 0

    docs = docs[:2]

    context = " ".join([d.page_content for d in docs])
    context = context[:600]

    prompt = f"""
Answer the question in 2-3 sentences using the context.

Context:
{context}

Question: {query}

Answer:
"""

    response = generator(prompt, do_sample=False)[0]["generated_text"]

    answer = response.split("Answer:")[-1].strip()

    return answer, len(docs)

In [11]:
def handle_query(query):
    result = ask(query)

    if result[0] is None:
        print("⚠️ No relevant data found. Escalating...")
        return input("👨‍💻 Enter human response: ")

    answer, doc_count = result

    if doc_count < 1:
        print("⚠️ Low confidence. Escalating...")
        return input("👨‍💻 Enter human response: ")

    return answer

In [12]:
while True:
    q = input("You: ")

    if q.lower() == "exit":
        break

    print("Bot:", handle_query(q))

You: What is R programming?
Bot: command-line-driven programming language


KeyboardInterrupt: Interrupted by user